In [ ]:
# CELL 1 - mount Drive
from google.colab import drive
drive.mount('/content/drive')
print("mounted")


In [ ]:
# ============================================================
#  WHAT DID THE RE-RUN ACTUALLY USE?
# ============================================================
import os, json, re, hashlib, pickle

ROOT = "/content/drive/MyDrive"

SRC = [
 "Phase1_Project/index_rebuild_base_verses_only/b6_build_index_and_retrieval_api.py",
 "Phase2_Project/Roma_output/src/b6_build_index_and_retrieval_api.py",
 "Phase2_Project/Roma_output/src/c1_cross_lingual_fallback.py",
 "Phase2_Project/Roma_output/src/c2_sufficiency_labels.py",
 "Phase2_Project/Roma_output/src/c3_query_refinement.py",
 "Phase3_Project/guardrail_output/src/b6_build_index_and_retrieval_api.py",
 "Phase3_Project/guardrail_output/src/d1_agent_loop.py",
 "Phase3_Project/guardrail_output/src/d2_cot_prompting.py",
 "Phase3_Project/guardrail_output/src/d3_sufficiency_and_fallback.py",
 "Phase3_Project/guardrail_output/src/e4_zero_hallucination_guardrail.py",
]
NEWFILES = [
 "Phase3_Project/guardrail_output/src/f1_calibrate_guardrail.py",
 "Phase3_Project/guardrail_output/src/f2_build_calibration_dataset.py",
]

PAT = re.compile(
    r"(model|MODEL|groq|Groq|llama|allam|gpt|qwen|mistral|gemma|"
    r"INDEX|index_dir|INDEX_DIR|BASE_DIR|\.pkl|\.hnsw|"
    r"threshold|THRESH|SUFFICIEN|MIN_SIM|max_seq|GATE|Omartificial|"
    r"sentence-transformers|from_pretrained|SentenceTransformer)")

def rd(rel):
    p = os.path.join(ROOT, rel)
    if not os.path.exists(p):
        return None
    return open(p, encoding="utf-8", errors="replace").read()

# ---------------------------------------------------------------- PART A
print("#" * 78)
print("# PART A - KEY LINES FROM EVERY SCRIPT THE RE-RUN TOUCHED")
print("#" * 78)
for rel in SRC:
    body = rd(rel)
    print(f"\n=== {rel} ===")
    if body is None:
        print("   MISSING")
        continue
    lines = body.splitlines()
    md5 = hashlib.md5(body.encode()).hexdigest()[:10]
    print(f"   {len(lines)} lines | md5 {md5}")
    shown = 0
    for i, ln in enumerate(lines, 1):
        s = ln.strip()
        if not s or s.startswith("#"):
            continue
        if PAT.search(ln):
            print(f"   {i:>4}: {ln.rstrip()[:150]}")
            shown += 1
            if shown >= 45:
                print("        ... (more matches suppressed)")
                break
    if shown == 0:
        print("   (no matching lines)")

# ---------------------------------------------------------------- PART B
print()
print("#" * 78)
print("# PART B - THE TWO BRAND-NEW SCRIPTS, IN FULL")
print("#" * 78)
for rel in NEWFILES:
    body = rd(rel)
    print(f"\n===== {rel} =====")
    if body is None:
        print("   MISSING")
        continue
    lines = body.splitlines()
    print(f"   ({len(lines)} lines)")
    for i, ln in enumerate(lines[:170], 1):
        print(f"   {i:>4}| {ln.rstrip()[:150]}")
    if len(lines) > 170:
        print(f"   ... +{len(lines)-170} more lines")

# ---------------------------------------------------------------- PART C
print()
print("#" * 78)
print("# PART C - WHAT IS INSIDE EACH INDEX")
print("#" * 78)
INDEXES = [
 "Phase1_Project/index_rebuild_base_verses_only/index",
 "Phase3_Project/guardrail_output_v2/index_v2",
 "Phase1_Project/MemberB_B4_B6_output/index",
 "Phase4_Project/index_chunked",
]
for rel in INDEXES:
    p = os.path.join(ROOT, rel, "entries.pkl")
    print(f"\n=== {rel} ===")
    if not os.path.exists(p):
        print("   no entries.pkl")
        continue
    try:
        with open(p, "rb") as f:
            ent = pickle.load(f)
    except Exception as e:
        print("   could not load:", e)
        continue
    print(f"   {len(ent):,} entries   type={type(ent).__name__}")
    if isinstance(ent, list) and ent and isinstance(ent[0], dict):
        print(f"   keys: {sorted(ent[0].keys())}")
        counts = {}
        for e in ent:
            counts[e.get("source_type", "?")] = counts.get(e.get("source_type", "?"), 0) + 1
        print(f"   source_type: {counts}")
        vks = {e.get("verse_key") for e in ent}
        print(f"   unique verse_keys: {len(vks):,}")
        t = str(ent[0].get("text", ""))[:120]
        print(f"   first text: {t}")
        lens = sorted(len(str(e.get('text',''))) for e in ent)
        print(f"   text length  median {lens[len(lens)//2]}  max {lens[-1]}")

# ---------------------------------------------------------------- PART D
print()
print("#" * 78)
print("# PART D - THE NEW PHASE 3 GUARDRAIL RUN, ALL ITEMS")
print("#" * 78)
p = os.path.join(ROOT, "Phase3_Project/guardrail_output/phase3_guardrail_results.json")
if os.path.exists(p):
    data = json.load(open(p, encoding="utf-8"))
    print(f"  {len(data)} items | keys: {sorted(data[0].keys())}\n")
    nver = 0
    for i, d in enumerate(data, 1):
        v = d.get("verified")
        nver += 1 if v else 0
        ctx = d.get("context") or []
        sims = [c.get("similarity") for c in ctx if isinstance(c, dict)]
        cited = set(re.findall(r"\[(\d+:\d+)\]", str(d.get("final_response", ""))))
        inctx = {c.get("verse_key") for c in ctx if isinstance(c, dict)}
        print(f"  {i:>2}. verified={v}  attempts={d.get('attempts')}  "
              f"stop={d.get('stopped_reason')}")
        print(f"      mismatch={d.get('final_mismatch_score')}  "
              f"ctx={len(ctx)}  topsim={max(sims) if sims else None}")
        print(f"      cited={sorted(cited)}")
        print(f"      cited NOT in context = {sorted(cited - inctx)}")
        print(f"      q: {str(d.get('query'))[:80]}")
    print(f"\n  VERIFIED: {nver}/{len(data)}")
else:
    print("  MISSING")

# ---------------------------------------------------------------- PART E
print()
print("#" * 78)
print("# PART E - THE NEW PHASE 2 INTEGRATION RUN, ALL ITEMS")
print("#" * 78)
p = os.path.join(ROOT, "Phase2_Project/Laiba_integration_output/test_query_results.json")
if os.path.exists(p):
    data = json.load(open(p, encoding="utf-8"))
    print(f"  {len(data)} items | keys: {sorted(data[0].keys())}\n")
    for i, d in enumerate(data, 1):
        ans = str(d.get("answer", ""))
        arabic = sum(1 for ch in ans if "؀" <= ch <= "ۿ")
        latin = sum(1 for ch in ans if ch.isascii() and ch.isalpha())
        lang = "ARABIC" if arabic > latin else ("ENGLISH" if latin > arabic else "?")
        print(f"  {i:>2}. rounds={d.get('rounds')}  stop={d.get('stopped_reason')}  "
              f"fallback={d.get('used_fallback')}  ground_ok={d.get('grounding_ok')}")
        print(f"      suff={d.get('sufficiency_score')}  answer_lang={lang}  "
              f"len={len(ans)}")
        print(f"      q: {str(d.get('query'))[:80]}")
    langs = []
    for d in data:
        a = str(d.get("answer", ""))
        ar = sum(1 for ch in a if "؀" <= ch <= "ۿ")
        la = sum(1 for ch in a if ch.isascii() and ch.isalpha())
        langs.append("AR" if ar > la else "EN")
    print(f"\n  answer languages: {langs.count('AR')} Arabic, {langs.count('EN')} English")
    print(f"  grounding_ok: {sum(1 for d in data if d.get('grounding_ok'))}/{len(data)}")
    print(f"  used_fallback: {sum(1 for d in data if d.get('used_fallback'))}/{len(data)}")
else:
    print("  MISSING")

# ---------------------------------------------------------------- PART F
print()
print("#" * 78)
print("# PART F - ANY GROQ / LLM MODEL STRING ANYWHERE IN THE NEW FOLDERS")
print("#" * 78)
SCAN = ["Phase2_Project", "Phase3_Project", "Phase1_Project/index_rebuild_base_verses_only"]
MODELPAT = re.compile(r"[\"']([a-z0-9][a-z0-9._/-]{4,60}(?:-\d+b|instruct|versatile|instant|"
                      r"turbo|preview|oss|allam|llama|qwen|gemma|mistral|gpt)[a-z0-9._/-]*)[\"']",
                      re.I)
seen = {}
for base in SCAN:
    for dp, dn, fns in os.walk(os.path.join(ROOT, base)):
        dn[:] = [d for d in dn if d not in {".git", "__pycache__", ".ipynb_checkpoints"}]
        for fn in fns:
            if not fn.endswith((".py", ".ipynb", ".json", ".txt", ".md")):
                continue
            fp = os.path.join(dp, fn)
            try:
                if os.path.getsize(fp) > 4_000_000:
                    continue
                body = open(fp, encoding="utf-8", errors="replace").read()
            except Exception:
                continue
            for m in MODELPAT.findall(body):
                if "/" in m or "-" in m:
                    seen.setdefault(m, set()).add(fp.replace(ROOT + "/", ""))
for m in sorted(seen):
    print(f"\n  {m}")
    for f in sorted(seen[m])[:4]:
        print(f"      {f}")
if not seen:
    print("\n  (no model string found)")

print("\n" + "#" * 78)
print("# DONE - paste everything above")
print("#" * 78)
